In [11]:
import os
import getpass
from dotenv import load_dotenv
load_dotenv(override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key:")
openrouter_model = os.getenv("MODEL", "openai/gpt-5-nano")

# Self-Ask

Self-Ask is a prompting method introduced in the October 2022 paper ["Measuring and Narrowing the Compositionality Gap in Language Models"](https://arxiv.org/pdf/2210.03350.pdf) by Ofir Press et al.

## Compositional reasoning

Compositional reasoning means answering a complex question by first answering smaller sub-questions, then combining those answers.

## The compositionality gap

The paper measures the compositionality gap as the share of questions where a model answers every sub-question correctly but still gets the combined question wrong. It tests whether a model can put facts together, not just recall them.

## What the paper found

- Larger models, such as GPT-3, improved more on single-hop questions (one fact needed) than on multi-hop questions (several linked facts needed).
- The compositionality gap did not shrink noticeably as models grew. Recalling more facts did not fix the problem of combining them.

## How Self-Ask works

1. Generate follow-up questions: instead of answering directly, the model asks itself simpler sub-questions.
2. Answer the follow-ups: each sub-question is answered, either by the model or by an external source such as a search engine.
3. Answer the original question: the model combines the intermediate answers into a final answer.

<figure>
  <img src="./assets/direct.png" alt="Diagram comparing direct prompting, chain of thought and Self-Ask on a question from the Bamboogle dataset" style="width:80%">
  <figcaption>Direct prompting (Brown et al., 2020) compared to chain of thought and self-ask method on a question from Bamboogle. Text with a white background is the prompt, text with a green background is the LM output, and underlined text is the inference-time question.</figcaption>
  <a href="https://ar5iv.labs.arxiv.org/html/2210.03350">Image Source</a>
</figure>

## Benefits

- It breaks a complex question into smaller steps, so the reasoning is easier to inspect.
- The follow-up questions give a clear point to plug in a search engine, so each sub-question can be answered from an external source.
- A helpful way to picture it: the model asks itself questions before it answers.

## Understanding the Self-Ask pseudo-code

The paper's authors published a demo notebook in their [GitHub repo](https://github.com/ofirpress/self-ask/blob/main/self-ask_plus_search-engine_demo.ipynb). The code below is simplified pseudo-code that shows the logic. It is not runnable as written, because `call_gpt` and `get_answer` are stubs that return empty strings.

### Key components

1. `self_ask`: the core function. It starts with the initial question, keeps generating and answering follow-up questions, and ends with a final answer.
2. Helper functions:
   - `frame_question`: prepares the initial question.
   - `call_gpt`: stands in for a call to a language model. Here it is a stub.
   - `get_last_line`: returns the last line of the text, where a follow-up or answer usually appears.
   - `extract_question`: pulls the follow-up question out of the text.
   - `get_answer`: stands in for fetching an answer from an external source. Here it is a stub.

### Workflow

- Frame the initial question.
- While the last line contains "Follow up:", extract the follow-up question and look for an answer.
- If an answer is found, add it to the prompt and call the model again.
- If no answer is found, ask the model for a final answer.
- Stop once a final answer is produced.

```python

def self_ask(question: str) -> str:
    cur_prompt = frame_question(question)  
    ret_text = call_gpt(cur_prompt)

    while "Follow up:" in get_last_line(ret_text):
        extracted_question = extract_question(ret_text)

        external_answer = get_answer(extracted_question)

        if external_answer:
            cur_prompt += "Intermediate answer: " + external_answer
            ret_text = call_gpt(cur_prompt)
        else:
            cur_prompt += "Intermediate answer: "
            ret_text = call_gpt(cur_prompt + " Follow up: " + "Final answer: ")

    if "Final answer:" not in ret_text:
        cur_prompt += "Final answer: "
        ret_text = call_gpt(cur_prompt)

    return cur_prompt + ret_text

def frame_question(question: str) -> str:
    return question

def call_gpt(prompt: str) -> str:
    return ""

def get_last_line(text: str) -> str:
    return text.split('\n')[-1] if '\n' in text else text

def extract_question(text: str) -> str:
    return text.split("Follow up:")[-1].strip()

def get_answer(question: str) -> str:
    return ""
```

# Self-Ask in LangChain

In [12]:
from langchain_openrouter import ChatOpenRouter
from langchain_community.tools import DuckDuckGoSearchRun

llm = ChatOpenRouter(model=openrouter_model, temperature=0)

search = DuckDuckGoSearchRun()

In [13]:
from langchain_core.tools import Tool 

tools = [
    Tool(
        name="Intermediate Answer",
        func=search.run,
        description="useful for when you need to ask with search",
    )
]

In [14]:
from langsmith import Client

client = Client(api_url=os.environ["LANGCHAIN_ENDPOINT"])

prompt = client.pull_prompt("moche/self-ask-with-search", dangerously_pull_public_prompt=True)


In [15]:
print(prompt.template)

Question: Who lived longer, Muhammad Ali or Alan Turing?
Are follow up questions needed here: Yes.
Follow up: How old was Muhammad Ali when he died?
Intermediate answer: Muhammad Ali was 74 years old when he died.
Follow up: How old was Alan Turing when he died?
Intermediate answer: Alan Turing was 41 years old when he died.
So the final answer is: Muhammad Ali

Question: When was the founder of craigslist born?
Are follow up questions needed here: Yes.
Follow up: Who was the founder of craigslist?
Intermediate answer: Craigslist was founded by Craig Newmark.
Follow up: When was Craig Newmark born?
Intermediate answer: Craig Newmark was born on December 6, 1952.
So the final answer is: December 6, 1952

Question: Who was the maternal grandfather of George Washington?
Are follow up questions needed here: Yes.
Follow up: Who was the mother of George Washington?
Intermediate answer: The mother of George Washington was Mary Ball Washington.
Follow up: Who was the father of Mary Ball Washin

# Few-shot example set (deep learning)

The next code cell defines a few-shot prompt. "Few-shot" means the prompt includes worked examples that show the model the format to follow. These examples are about deep learning, not the Bamboogle-style questions in the prompt pulled from LangSmith.

The layout uses "Follow-up 1", "Intermediate Answer" and "Final Answer". That differs from the "Follow up:" format used by the pulled prompt. Later cells do not use this template, so treat it as an example to adapt.

Some example answers simplify the research. For instance, "Adam converges faster" is an oversimplification, so check claims before reusing the examples.

In [16]:
from langchain_core.prompts import PromptTemplate

deep_learning_template = """
Question: Which architecture is more suited for image classification, LSTM or CNN?
Are follow-up questions needed here: Yes.
Follow-up 1: What is the primary application of LSTM?
Intermediate Answer: LSTM (Long Short-Term Memory) is primarily used for sequential data and time series analysis.
Follow-up 2: What is the primary application of CNN?
Intermediate Answer: CNN (Convolutional Neural Network) is primarily used for image and video recognition.
Final Answer: CNN is more suited for image classification.

Question: Which optimizer converges faster, SGD or Adam?
Are follow-up questions needed here: Yes.
Follow-up 1: What is the main advantage of SGD (Stochastic Gradient Descent)?
Intermediate Answer: SGD is simple and easy to implement.
Follow-up 2: What is the main advantage of Adam optimizer?
Intermediate Answer: Adam combines the benefits of both AdaGrad and RMSProp and usually converges faster than SGD.
Final Answer: Adam converges faster.

Question: For natural language processing tasks, which dataset is larger, MNIST or WikiText-103?
Are follow-up questions needed here: Yes.
Follow-up 1: What kind of data does the MNIST dataset contain?
Intermediate Answer: MNIST contains handwritten digits and is used for image classification.
Follow-up 2: What kind of data does the WikiText-103 dataset contain?
Intermediate Answer: WikiText-103 is a large language modeling dataset containing text from Wikipedia articles.
Final Answer: WikiText-103 is larger and is used for natural language processing tasks.

Question: {input}
Are followup questions needed here:{agent_scratchpad}
"""

deep_learning_prompt = PromptTemplate.from_template(deep_learning_template)

# Binding a stop sequence

A stop sequence is a string that tells the model to stop generating text when it appears. In this notebook the stop sequence is `"Intermediate answer:"`, set in the manual chain code with `llm.bind(stop=["Intermediate answer:"])`.

## Purpose in Self-Ask with a search agent

- The model writes a follow-up question and then the text `Intermediate answer:`.
- The stop sequence halts generation at that point, so the model does not invent the answer.
- The search tool supplies the answer, which is added to the prompt, and the model continues.

## How the stop sequence works

- It acts as a delimiter in the model's output.
- Once the model is about to produce it, generation ends and control returns to the code.
- Whether the stop text itself appears in the returned output can vary by model provider, so check your output.

<figure>
  <img src="https://ar5iv.labs.arxiv.org/html/2210.03350/assets/x9.png" alt="Diagram of the Self-Ask loop with a search engine: the model writes a follow-up question, the search engine answers it, and the answer is added back to the prompt" style="width:100%">
  <figcaption>Self-ask + Search Engine: prompt on white background, LM-generated text in green. Start by using a few-shot prompt and appending the test-time question (underlined) to it. The LM then generates a follow-up question which inputs to an internet search engine. The response is inserted back into the rest of the prompt to let the LM generate the next follow-up question. This process then repeats until the LM decides to output the final answer.</figcaption>
  <a href="https://ar5iv.labs.arxiv.org/html/2210.03350">Image Source</a>
</figure>

# Manual Self-Ask chain

The next code cells build the Self-Ask agent by hand. They are commented out, so remove the `#` characters to run them. The first defines the chain, which combines the prompt, the model with its stop sequence, and an output parser. The two after it call the chain with one question each.

In [17]:
# from langchain_classic.agents import AgentExecutor
# from langchain_classic.agents.output_parsers import SelfAskOutputParser
# from langchain_classic.agents.format_scratchpad import format_log_to_str

# llm_with_stop = llm.bind(stop=["Intermediate answer:"])

# agent = {
#     "input": lambda x: x["input"],
#     # Use some custom observation_prefix/llm_prefix for formatting
#     "agent_scratchpad": lambda x: format_log_to_str(
#         x['intermediate_steps'],
#         observation_prefix="Intermediate answer: ",
#         llm_prefix="",
#     ),
# } | prompt | llm_with_stop | SelfAskOutputParser()

# agent_executor = AgentExecutor(agent=agent,
#                                tools=tools,
#                                verbose=True,
#                                handle_parsing_errors=True,
#                                max_iterations=5
#                                )

In [18]:
# agent_executor.invoke({"input": "What is the RWKV architecture for LLMs?"})

In [19]:
# agent_executor.invoke({"input": "How is Direct Preference Optimization different than traditional RLHF?"})

# What the outputs show

The RWKV and DPO code cells have saved outputs from an earlier run. If you run them again, the results may differ.

## RWKV

- The model wrote its own "Intermediate answer" lines and never called the search tool. The stop sequence is meant to prevent this, so the saved output suggests it was not in effect for that run.
- The answer is wrong. RWKV is short for Receptance Weighted Key Value. The model gave a confident, plausible-sounding definition, which is a hallucination.

## DPO

- The answer is broadly reasonable but simplified. Direct Preference Optimization trains a model directly on preference pairs, without a separate reward model or a reinforcement learning loop.
- It also wrote its own intermediate answers, so treat it as a starting point to check, not a verified source.

# Alternative: the helper function

The manual chain wires the prompt, the model and an output parser into a chain, then runs it with `AgentExecutor`. The final code cell uses `create_self_ask_with_search_agent`, a LangChain helper that builds that agent for you.

The final code cell also truncates each search result to 1000 characters, so long pages do not fill the prompt. Like the manual chain, it is commented out and has no saved output. It reuses the `llm` and the prompt pulled from LangSmith.

In [20]:
# from langchain_classic.agents import AgentExecutor, create_self_ask_with_search_agent

# # Reuses `llm` (cell 2) and `prompt` (cell 4, pulled from LangSmith)

# tools = [
#     Tool(
#         name="Intermediate Answer",
#         func=lambda q: search.run(q)[:1000],  # truncate long search results
#         description="useful for when you need to ask with search",
#     )
# ]

# agent = create_self_ask_with_search_agent(llm, tools, prompt)

# agent_executor = AgentExecutor(agent=agent,
#                                tools=tools,
#                                verbose=True,
#                                handle_parsing_errors=True,
#                                max_iterations=5)

# agent_executor.invoke({"input": "Who is Hippie Sabotage and where are they from?"})

# Limitations and responsible use

- Hallucinated sub-answers: if the model answers its own follow-up questions without a search, it can state false facts confidently (see the RWKV example above).
- Noisy search results: snippets can be irrelevant, out of date or partly wrong. Truncating them keeps the prompt short but does not make them more accurate.
- Cost and speed: each follow-up is an extra model call and possibly an extra search, so one question can take several calls.
- Privacy: search queries go to a third-party search service. Do not include personal or confidential details in questions.
- Verify sources: check the final answer against the sources it relied on.
- Tool status: the DuckDuckGo search tool comes from `langchain_community`, which the warning in the setup cell says is no longer maintained. Free search services may also rate-limit requests.